# **Experiment-5: Subword Level Tokenization and POS Tagging**

This experiment covers:
- **5.1**: BPE Tokenization (pretrained & from scratch)
- **5.2**: SentencePiece Tokenization (pretrained & from scratch)
- **5.3**: POS Tagging (spaCy & NLTK)
- **5.4**: POS Tagging with Token Frequency

In [6]:
pip install --upgrade typing_extensions pydantic pydantic-core

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: typing_extensions in c:\Program Files\Python312\Lib\site-packages (4.16.0)




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
# IMPORTS
# ============================================================

# BPE Tokenization (HuggingFace tokenizers library)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Pretrained tokenizers (HuggingFace transformers)
from transformers import AutoTokenizer

# SentencePiece tokenizer (Google)
import sentencepiece as spm

# POS Tagging
import spacy
import nltk
from collections import Counter

# Download required NLTK data files
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

c:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91976\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\91976\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\91976\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\91976\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [2]:
# ============================================================
# LOAD INPUT DATA
# ============================================================

file_path = "input_sub_word_data.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# Show a preview of the corpus
print("Input Data (first 500 characters):")
print("=" * 50)
print(text[:500])
print("...")

Input Data (first 500 characters):
Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is 
...


In [3]:
# ============================================================
# HELPER FUNCTION: Display tokens neatly
# ============================================================

def display_tokens(tokens, token_ids, title, max_display=20):
    """Print tokens and their IDs in a formatted table.
    
    Args:
        tokens: List of token strings
        token_ids: List of corresponding integer IDs
        title: Section heading to print
        max_display: Number of tokens to show (default 20)
    """
    print(f"\n{title}")
    print(f"{'Token':30} {'Token ID'}")
    print("-" * 45)

    for token, tid in list(zip(tokens, token_ids))[:max_display]:
        print(f"{token:30} {tid}")

    if len(tokens) > max_display:
        print(f"\n... ({len(tokens) - max_display} more tokens not shown)")

    print(f"\nTotal tokens: {len(tokens)}")
    print(f"Unique tokens: {len(set(tokens))}")

---
## Experiment 5.1: BPE Tokenization

Tokenize text into subword units using BPE. Display tokens and their IDs.

**Two approaches:** (a) Pretrained model, (b) Trained from scratch

In [5]:
# ============================================================
# 5.1(a) BPE — Using Pretrained GPT-2 Tokenizer
# ============================================================
# GPT-2's tokenizer was trained on ~40GB of internet text.
# It has a vocabulary of ~50,257 subword tokens.
# The 'G' prefix (shown as Ġ) means there's a leading space.

bpe_tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokens = bpe_tokenizer.tokenize(text)
token_ids = bpe_tokenizer.convert_tokens_to_ids(tokens)

display_tokens(tokens, token_ids, "BPE Tokenization — Pretrained GPT-2")

Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors



BPE Tokenization — Pretrained GPT-2
Token                          Token ID
---------------------------------------------
Natural                        35364
Ġlanguage                      3303
Ġprocessing                    7587
Ġis                            318
Ġa                             257
Ġbranch                        8478
Ġof                            286
Ġartificial                    11666
Ġintelligence                  4430
Ġthat                          326
Ċ                              198
en                             268
ables                          2977
Ġcomputers                     9061
Ġto                            284
Ġunderstand                    1833
,                              11
Ġinterpret                     6179
,                              11
Ġand                           290

... (1328 more tokens not shown)

Total tokens: 1348
Unique tokens: 481


In [ ]:
# ============================================================
# 5.1(b) BPE — Trained from Scratch on Our Corpus
# ============================================================
# We train a BPE tokenizer with vocab_size=500 on our small corpus.
# min_frequency=2 means only character pairs appearing >=2 times get merged.

bpe = Tokenizer(BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)

bpe.train(files=[file_path], trainer=trainer)

output = bpe.encode(text)

display_tokens(output.tokens, output.ids, "BPE Tokenization — Trained from Scratch")
print(f"Vocabulary Size: {bpe.get_vocab_size()}")


BPE Tokenization — Trained from Scratch
Token                          Token ID
---------------------------------------------
Natural                        359
language                       142
processing                     192
is                             82
a                              24
b                              25
r                              41
an                             54
ch                             84
of                             89
ar                             59
t                              43
ific                           444
ial                            380
in                             51
t                              43
el                             105
l                              35
ig                             307
ence                           178

... (1588 more tokens not shown)

Total tokens: 1608
Unique tokens: 394
Vocabulary Size: 500


: 

---
## Experiment 5.2: SentencePiece Tokenization

Tokenize text into subword units using SentencePiece. Display tokens and their IDs.

**Two approaches:** (a) Pretrained model, (b) Trained from scratch

In [ ]:
# ============================================================
# 5.2(a) SentencePiece — Using Pretrained mT5-small
# ============================================================
# mT5 (multilingual T5) uses SentencePiece with 250,000 tokens.
# The '▁' prefix represents a leading space.

sp_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

sp_tokens = sp_tokenizer.tokenize(text)
sp_token_ids = sp_tokenizer.convert_tokens_to_ids(sp_tokens)

display_tokens(sp_tokens, sp_token_ids, "SentencePiece Tokenization — Pretrained mT5")

In [ ]:
# ============================================================
# 5.2(b) SentencePiece — Trained from Scratch
# ============================================================
# We train our own SentencePiece model with vocab_size=300.
# character_coverage=1.0 means include all characters from the corpus.

spm.SentencePieceTrainer.train(
    input=file_path,
    model_prefix="my_sentencepiece",
    vocab_size=300,
    model_type="bpe",
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor(model_file="my_sentencepiece.model")

sp_tokens = sp.encode(text, out_type=str)
sp_token_ids = sp.encode(text, out_type=int)

display_tokens(sp_tokens, sp_token_ids, "SentencePiece Tokenization — Trained from Scratch")
print(f"Vocabulary Size: {sp.get_piece_size()}")

---
## Experiment 5.3: POS Tagging

Perform Part-of-Speech tagging on:

> *"The young student is reading an interesting book in the library."*

**Two approaches:** (a) spaCy, (b) NLTK

In [ ]:
# Define the sentence for POS tagging
pos_text = "The young student is reading an interesting book in the library."

In [ ]:
# ============================================================
# 5.3(a) POS Tagging — Using spaCy
# ============================================================
# spaCy uses a neural network model for POS tagging.
# .pos_ gives the Universal POS tag (NOUN, VERB, ADJ, etc.)
# spacy.explain() converts the tag to a readable description.

nlp = spacy.load("en_core_web_sm")
doc = nlp(pos_text)

print("POS Tagging using spaCy\n")
print(f"{'Token':20} {'POS Tag':10} {'Description'}")
print("-" * 60)

for token in doc:
    print(f"{token.text:20} {token.pos_:10} {spacy.explain(token.pos_)}")

In [ ]:
# ============================================================
# 5.3(b) POS Tagging — Using NLTK
# ============================================================
# NLTK uses the Averaged Perceptron tagger.
# It gives Penn Treebank tags (DT, JJ, NN, VBZ, VBG, IN, etc.)

nltk_tokens = nltk.word_tokenize(pos_text)
nltk_tags = nltk.pos_tag(nltk_tokens)

print("POS Tagging using NLTK\n")
print(f"{'Token':20} {'POS Tag'}")
print("-" * 40)

for token, tag in nltk_tags:
    print(f"{token:20} {tag}")

---
## Experiment 5.4: POS Tagging with Token Frequency

Tag the full corpus with POS tags and display each unique token's tag, description, and frequency.

In [ ]:
# ============================================================
# 5.4 POS Tagging with Frequency — Using spaCy
# ============================================================

doc = nlp(text)

# Count how often each lowercased token appears (skip whitespace)
frequency = Counter(
    token.text.lower()
    for token in doc
    if not token.is_space
)

# Collect unique token info (POS, description, frequency)
token_info = {}
for token in doc:
    if token.is_space:
        continue
    word = token.text.lower()
    if word not in token_info:
        token_info[word] = {
            "POS": token.pos_,
            "Description": spacy.explain(token.pos_),
            "Frequency": frequency[word]
        }

# Display results
print("POS Tagging with Frequency — spaCy\n")
print(f"{'Token':20} {'POS Tag':10} {'Description':20} {'Frequency'}")
print("-" * 75)

for word, info in token_info.items():
    print(
        f"{word:20} "
        f"{info['POS']:10} "
        f"{str(info['Description']):20} "
        f"{info['Frequency']}"
    )

print(f"\nTotal unique tokens: {len(token_info)}")